In [3]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [3]:
expr_df = pd.read_csv("/cluster/work/boeva/eheiss/datasets/GDSC/drug_response_expr_data.csv", index_col=0, low_memory=False)
expr_df.index = expr_df.index.astype(str)
print(expr_df.shape)
expr_df.head()

(700, 19104)


,cell_line_display_name,lineage_1,lineage_2,lineage_3,lineage_6,lineage_4,NEMP2,SPDYE11,MED6,SPATA1,...,XYLB,CDC25A,NR1H4,NUP153,SUPT7L,GFPT2,USP15,IQSEC1,FGFBP1,FGF19
depmap_id,,,,,,,,,,,,,,,,,,,,,
ACH-000973,639V,Bladder/Urinary Tract,Urethral Cancer,Urethral Urothelial Carcinoma,NaN,NaN,2.350843,0.028249,5.144697,0.501200,...,2.911525,4.919169,-0.004244,5.058834,4.850728,3.627488,5.157255,3.509118,0.619068,-0.012049
ACH-000757,A427,Lung,Non-Small Cell Lung Cancer,Lung Adenocarcinoma,NaN,NaN,2.291779,-0.007359,5.119924,1.057133,...,2.174921,3.924597,-0.004244,4.806072,4.504631,2.145680,5.489351,2.646765,0.068688,0.014902
ACH-000248,AU565,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,HER2+,NaN,0.665334,-0.007359,5.392927,0.767300,...,2.852315,3.967462,0.036676,5.313185,4.651704,0.690699,5.182693,4.079506,0.205939,-0.012049
ACH-001016,BECKER,CNS/Brain,Diffuse Glioma,Astrocytoma,NaN,NaN,1.581257,0.016474,5.572271,0.146312,...,1.085777,3.120451,0.165230,4.544086,5.531915,4.626524,5.355187,2.052331,0.291948,1.389632
ACH-000245,BL41,Lymphoid,Mature B-Cell Neoplasms,Burkitt Lymphoma,NaN,NaN,2.194331,-0.007359,5.231515,0.482211,...,2.311855,5.645543,-0.004244,5.460451,4.136652,0.102620,5.233888,3.945063,0.183967,0.054353


In [4]:
gene_info = pd.read_csv("/cluster/work/boeva/eheiss/scbFM/data/bulkformer_gene_info.csv")
sym2ensg = dict(zip(gene_info["gene_symbol"].astype(str), gene_info["ensg_id"].astype(str)))

symbol_cols = [c for c in expr_df.columns if c in sym2ensg]
ensg_ids = [sym2ensg[c] for c in symbol_cols]

expr_mapped = expr_df[symbol_cols].copy()
expr_mapped.columns = ensg_ids
expr_mapped = expr_mapped.loc[:, ~expr_mapped.columns.duplicated(keep="first")]

n_mapped = len(expr_mapped.columns)
n_unmapped = len(expr_df.columns) - len(symbol_cols)
print(f"Mapped: {n_mapped} / {len(expr_df.columns)}  |  Unmapped: {n_unmapped}")


Mapped: 18916 / 19104  |  Unmapped: 188


In [5]:
adata = ad.AnnData(X=expr_mapped.values.astype("float32"))
adata.obs_names = list(expr_mapped.index)
adata.var_names = list(expr_mapped.columns)
adata

AnnData object with n_obs × n_vars = 700 × 18916

In [6]:
ic50 = pd.read_csv("/cluster/work/boeva/eheiss/datasets/GDSC/drug_response_prediction_IC50.csv")
ic50_cell_ids = set(ic50["ModelID"].astype(str))
expr_cell_ids = set(adata.obs_names)
overlap = ic50_cell_ids & expr_cell_ids
print(f"Cell lines in IC50: {len(ic50_cell_ids)}")
print(f"Cell lines in expression: {len(expr_cell_ids)}")
print(f"Overlap: {len(overlap)}")


Cell lines in IC50: 700
Cell lines in expression: 700
Overlap: 700


In [12]:
adata.write("/cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad")


## Part II - Statistics

In [1]:
with open("/cluster/work/boeva/eheiss/scbFM/data/gene_list.txt") as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [4]:
gdsc = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad")
print(gdsc)

AnnData object with n_obs × n_vars = 700 × 18916


In [5]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(gdsc.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, gdsc.n_obs, chunk_size):
    end = min(start + chunk_size, gdsc.n_obs)

    X_chunk = gdsc.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == gdsc.n_obs:
        print(f"Processed {n_obs_done:,}/{gdsc.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


Processed 700/700 samples
Average portion of non-zero genes NOT in gene_list: 0.24909700360122852
Average portion of total reads NOT in gene_list: 0.1584323325753212


## Part III - filter to gene list

In [6]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(gdsc.var_names.astype(str))

reorder_idx = gene_index.get_indexer(gene_list)
missing = [g for g, i in zip(gene_list, reorder_idx) if i < 0]
if missing:
    raise ValueError(f"{len(missing)} genes from gene_list are missing in gdsc. First 20: {missing[:20]}")

out_path = "/cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad"
chunk_size = 1000

chunks = []

for start in range(0, gdsc.n_obs, chunk_size):
    end = min(start + chunk_size, gdsc.n_obs)
    X_chunk = gdsc.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC handles column selection more reliably than CSR on some SciPy builds.
        X_chunk = X_chunk.tocsc()[:, reorder_idx].tocsr()
    else:
        X_chunk = np.asarray(X_chunk)[:, reorder_idx]

    chunk = ad.AnnData(
        X=X_chunk,
        obs=gdsc.obs.iloc[start:end].copy(),
        var=pd.DataFrame(index=pd.Index(gene_list, name=gdsc.var_names.name)),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{gdsc.n_obs:,} samples")

gdsc_aligned = ad.concat(chunks, axis=0, join="inner", merge="same")
gdsc_aligned.var_names = pd.Index(gene_list, name=gdsc.var_names.name)

gdsc_aligned.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", gdsc_aligned.shape)

Prepared 700/700 samples
Wrote: /cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad
Shape: (700, 13004)
